In [1]:
import cv2
import numpy as np

In [39]:
def getThrowingArm(keypoints):
    rel_x_right = keypoints[:, 10, 0] - keypoints[:, 6, 0]
    rel_x_left  = keypoints[:, 9, 0] - keypoints[:, 5, 0]

    for arr in [rel_x_right, rel_x_left]:
        if np.any(np.isnan(arr)):
            valid = ~np.isnan(arr)
            arr[:] = np.interp(np.arange(len(arr)), np.where(valid)[0], arr[valid])

    range_right = np.nanmax(rel_x_right) - np.nanmin(rel_x_right)
    range_left  = np.nanmax(rel_x_left)  - np.nanmin(rel_x_left)
    return "right" if range_right > range_left else "left"

In [40]:
def getForwardDirection(rel_x):
    start = int(0.1 * len(rel_x))
    end   = int(0.9 * len(rel_x))
    if end <= start:
        return 1
    start_mean = np.nanmean(rel_x[:start])
    end_mean   = np.nanmean(rel_x[end:])
    return 1 if end_mean > start_mean else -1

In [4]:
def findPitchStart(keypoints):
    arm = detect_throwing_arm(keypoints)
    shoulder_idx, wrist_idx = (6, 10) if arm = "right" else (5, 9)
    shoulder_x = keypoints[:, shoulder_idx, 0]
    wrist_x    = keypoints[:, wrist_idx, 0]
    rel_x = wrist_x - shoulder_x

    if np.any(np.isnan(rel_x)):
        valid = ~np.isnan(rel_x)
        rel_x = np.interp(np.arange(len(rel_x)), np.where(valid)[0], rel_x[valid])
    direction = get_forward_direction(rel_x)
    signed_rel = direction * rel_x
    
    kernel = np.ones(window := 5) / window
    signed_rel_smooth = np.convolve(signed_rel, kernel, mode="same")
    if len(signed_rel_smooth) <= (search_start := 30):
        return 0
    min_idx = np.argmin(signed_rel_smooth[search_start:]) + search_start

    range_after = np.max(signed_rel_smooth[min_idx:]) - np.min(signed_rel_smooth[min_idx:])
    threshold = 0.05 * range_after
    for i in range(min_idx + 1, len(signed_rel_smooth)):
        if signed_rel_smooth[i] > signed_rel_smooth[min_idx] + threshold:
            if i + 3 < len(signed_rel_smooth) and all(
                signed_rel_smooth[j] > signed_rel_smooth[j-1] for j in range(i+1, i+4)
            ):
                return i
            return i
    return min_idx

In [41]:
def visualize(keypoints, start, video, save):
    cap = cv.VideoCapture(video)
    fourcc = cv.VideoWriter_fourcc(*"mp4v")
    fps = int(cap.get(cv.CAP_PROP_FPS))
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    out = cv.VideoWriter(save, fourcc, fps, (width, height))
    
    skeleton = [(5,7), (7,9), (6,8), (8,10), (5,6), (5,11), (6,12), (11,13), (13,15), (12,14), (14,16)]
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_array):
            break

        kp = keypoints_array[frame_idx]
        if not np.isnan(kp).all():
            for i, (x, y) in enumerate(kp):
                if not (np.isnan(x) or np.isnan(y)):
                    cv.circle(frame, (int(x), int(y)), 3, (0, 255, 0), -1)
            for (i, j) in skeleton:
                if not (np.isnan(kp[i,0]) or np.isnan(kp[i,1]) or np.isnan(kp[j,0]) or np.isnan(kp[j,1])):
                    pt1 = (int(kp[i, 0]), int(kp[i, 1]))
                    pt2 = (int(kp[j, 0]), int(kp[j, 1]))
                    cv.line(frame, pt1, pt2, (255, 0, 0), 2)

        if frame_idx == start_idx:
            cv.putText(frame, "START", (50,50), cv.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            cv.rectangle(frame, (0,0), (width-1, height-1), (0,0,255), 5)
        out.write(frame)
        frame_idx += 1
    
    cap.release()
    out.release()

In [ ]:
keypoints = np.load(KEYPOINT_PATH)
start = findPitchStart(keypoints)
truncated = keypoints[:(start + 1)]
visualize(keypoints, start, VIDEO_PATH, SAVE_PATH)